In [14]:
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
import mlflow

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
client = MlflowClient(MLFLOW_TRACKING_URI)

### Searching runs and experiments

In [6]:
client.search_experiments()

[<Experiment: artifact_location='file:///c:/Users/urbii/Desktop/Projekty/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1720033089110, experiment_id='1', last_update_time=1720033089110, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1720032974085, experiment_id='0', last_update_time=1720032974085, lifecycle_stage='active', name='Default', tags={}>]

In [7]:
client.create_experiment(name="new-experiment")

'2'

In [12]:
runs = client.search_runs(
    experiment_ids=1,
    filter_string="metric.rmse < 6.4",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [13]:
for run in runs:
    print(f"run id: {run.info.run_id},rmse: {run.data.metrics['rmse']:.4f}")

run id: 2343c74ff67a4ad885cbe3a3bbde5230,rmse: 6.2997
run id: 7ba48818e4ed47a9a1c546bbcc97133d,rmse: 6.2997
run id: 1f9ecbc1adf9462ca3c5122a23814799,rmse: 6.2997
run id: 0e885869e8264168b358b39f78263e7a,rmse: 6.3040
run id: 561e0629d8ca476d8212f84703cfeec8,rmse: 6.3054


### Promote model to model registry

In [15]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
run_id = '0e885869e8264168b358b39f78263e7a'
model_uri = f"runs:/{run_id}/model"

mlflow.register_model(model_uri=model_uri, name='nyc-taxi-regressor')

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '3' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1720127648997, current_stage='None', description=None, last_updated_timestamp=1720127648997, name='nyc-taxi-regressor', run_id='0e885869e8264168b358b39f78263e7a', run_link=None, source='file:///c:/Users/urbii/Desktop/Projekty/mlops-zoomcamp/02-experiment-tracking/mlruns/1/0e885869e8264168b358b39f78263e7a/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [16]:
client.search_registered_models()

[<RegisteredModel: aliases={'challenger': 1, 'champion': 2}, creation_timestamp=1720109020378, description='', last_updated_timestamp=1720127648997, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1720127648997, current_stage='None', description=None, last_updated_timestamp=1720127648997, name='nyc-taxi-regressor', run_id='0e885869e8264168b358b39f78263e7a', run_link=None, source='file:///c:/Users/urbii/Desktop/Projekty/mlops-zoomcamp/02-experiment-tracking/mlruns/1/0e885869e8264168b358b39f78263e7a/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=3>], name='nyc-taxi-regressor', tags={}>]

In [25]:
client.get_model_version_by_alias(name="nyc-taxi-regressor",alias="champion")

<ModelVersion: aliases=['champion'], creation_timestamp=1720109096606, current_stage='None', description='', last_updated_timestamp=1720109096606, name='nyc-taxi-regressor', run_id='65d7cd1682a94e0c8eb699c039bd09f6', run_link='', source='file:///c:/Users/urbii/Desktop/Projekty/mlops-zoomcamp/02-experiment-tracking/mlruns/1/65d7cd1682a94e0c8eb699c039bd09f6/artifacts/model', status='READY', status_message=None, tags={'model': 'linear_regression'}, user_id=None, version=2>

In [29]:
client.set_registered_model_alias(name="nyc-taxi-regressor", alias="challenger", version=3)

In [30]:
client.set_registered_model_alias(name="nyc-taxi-regressor", alias="staging", version=1)